In [4]:
#pip install -q datasets tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
#checking all the datasets in wikipedia
from datasets import get_dataset_config_names

configs = get_dataset_config_names("wikimedia/wikipedia")
print(f"Number of configs: {len(configs)}")
print(configs[:50])  # print first 50

/opt/anaconda3/envs/pyg310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of configs: 323
['20231101.ab', '20231101.ace', '20231101.ady', '20231101.af', '20231101.als', '20231101.alt', '20231101.am', '20231101.ami', '20231101.an', '20231101.ang', '20231101.anp', '20231101.ar', '20231101.arc', '20231101.ary', '20231101.arz', '20231101.as', '20231101.ast', '20231101.atj', '20231101.av', '20231101.avk', '20231101.awa', '20231101.ay', '20231101.az', '20231101.azb', '20231101.ba', '20231101.ban', '20231101.bar', '20231101.bat-smg', '20231101.bcl', '20231101.be', '20231101.be-x-old', '20231101.bg', '20231101.bh', '20231101.bi', '20231101.bjn', '20231101.blk', '20231101.bm', '20231101.bn', '20231101.bo', '20231101.bpy', '20231101.br', '20231101.bs', '20231101.bug', '20231101.bxr', '20231101.ca', '20231101.cbk-zam', '20231101.cdo', '20231101.ce', '20231101.ceb', '20231101.ch']


In [2]:
#extracting sv socs
sv_configs = [c for c in configs if ".sv" in c or c.endswith("sv") or "sv" in c]
sv_configs[:20]

['20231101.sv']

In [3]:
from datasets import load_dataset

config_name = "20231101.sv"   # change if needed after inspecting configs
ds = load_dataset("wikimedia/wikipedia", config_name, split="train")
ds

Generating train split: 100%|█| 2574513/2574513 [00:02<00:00, 998554.47 examples


Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 2574513
})

In [4]:
print(ds.column_names)
print(ds[0])

['id', 'url', 'title', 'text']
{'id': '1', 'url': 'https://sv.wikipedia.org/wiki/Amager', 'title': 'Amager', 'text': 'Amager är en dansk ö i Öresund. Öns norra och västra delar tillhör Köpenhamn, medan övriga delar upptas av Tårnby kommun och Dragørs kommun. \nAmager har en yta på 96,29 km² och befolkningen uppgår till 196\xa0047 personer (1/1 2018). På den östra delen av ön ligger Köpenhamns flygplats (Kastrup). Amager har flera militära anläggningar, såsom Kastrup fort och Dragör fort. På ön finns strandområdena Amager Strandpark och Kastrup Strandpark, flera restauranger i världsklass, det stora naturområdet Amager Fælled, ett av Nordens största köpcentra (Field\'s) och mycket annat. Amager förkortas ofta Ama\'r för att återspegla det danska uttalet.\n\nAmager är delvis en konstgjord ö, delvis en naturlig sådan. Till exempel anlades en konstgjord ö i anslutning till den redan existerande Amager Strandpark, när denna rustades upp väsentligen 2005. Ön är mycket låglänt och vissa delar

In [5]:
sample = ds[0]
for k, v in sample.items():
    print(f"\n--- {k} ---")
    print(str(v)[:500])


--- id ---
1

--- url ---
https://sv.wikipedia.org/wiki/Amager

--- title ---
Amager

--- text ---
Amager är en dansk ö i Öresund. Öns norra och västra delar tillhör Köpenhamn, medan övriga delar upptas av Tårnby kommun och Dragørs kommun. 
Amager har en yta på 96,29 km² och befolkningen uppgår till 196 047 personer (1/1 2018). På den östra delen av ön ligger Köpenhamns flygplats (Kastrup). Amager har flera militära anläggningar, såsom Kastrup fort och Dragör fort. På ön finns strandområdena Amager Strandpark och Kastrup Strandpark, flera restauranger i världsklass, det stora naturområdet Ama


In [6]:
#cleaning the text
import re
from tqdm.auto import tqdm

def clean_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    return text

texts = []

for ex in tqdm(ds):
    text = ex.get("text", "")
    if not text:
        continue
    text = clean_text(text)
    if len(text) < 200:   # skip very short entries
        continue
    texts.append(text)

print(f"Collected {len(texts)} cleaned articles")
print(texts[0][:1000])

100%|██████████████████████████████| 2574513/2574513 [01:53<00:00, 22592.97it/s]

Collected 2397805 cleaned articles
Amager är en dansk ö i Öresund. Öns norra och västra delar tillhör Köpenhamn, medan övriga delar upptas av Tårnby kommun och Dragørs kommun. Amager har en yta på 96,29 km² och befolkningen uppgår till 196 047 personer (1/1 2018). På den östra delen av ön ligger Köpenhamns flygplats (Kastrup). Amager har flera militära anläggningar, såsom Kastrup fort och Dragör fort. På ön finns strandområdena Amager Strandpark och Kastrup Strandpark, flera restauranger i världsklass, det stora naturområdet Amager Fælled, ett av Nordens största köpcentra (Field's) och mycket annat. Amager förkortas ofta Ama'r för att återspegla det danska uttalet. Amager är delvis en konstgjord ö, delvis en naturlig sådan. Till exempel anlades en konstgjord ö i anslutning till den redan existerande Amager Strandpark, när denna rustades upp väsentligen 2005. Ön är mycket låglänt och vissa delar ligger under havsytan, framför allt det genom fördämning och utfyllnad skapade området Kalve

In [7]:
#saving clean file
output_file = "swedish_wikipedia_clean.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for t in texts:
        f.write(t + "\n")

print(f"Saved to {output_file}")

Saved to swedish_wikipedia_clean.txt


In [12]:
#split data in train, val and test
from sklearn.model_selection import train_test_split

output_file = "swedish_wikipedia_clean.txt"

# Read the cleaned text file
with open(output_file, "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f if line.strip()]

train_texts, temp_texts = train_test_split(texts, test_size=0.02, random_state=42)
val_texts, test_texts = train_test_split(temp_texts, test_size=0.5, random_state=42)

for fname, split_data in [
    ("train.txt", train_texts),
    ("val.txt", val_texts),
    ("test.txt", test_texts),
]:
    with open(fname, "w", encoding="utf-8") as f:
        for item in split_data:
            f.write(item + "\n")

In [14]:
#pip install -q transformers datasets tokenizers accelerate sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [18]:
#train tokenizer
from tokenizers import ByteLevelBPETokenizer
import os

tokenizer_dir = "sv_tokenizer"
os.makedirs(tokenizer_dir, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=["train.txt"],
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>"]
)

tokenizer.save_model(tokenizer_dir)

print("Tokenizer saved to:", tokenizer_dir)
print(os.listdir(tokenizer_dir))




Tokenizer saved to: sv_tokenizer
['merges.txt', 'vocab.json']


In [13]:
#check length of trainer tokenizer
from transformers import GPT2TokenizerFast

tokenizer_dir = "sv_tokenizer"
hf_tokenizer = GPT2TokenizerFast(
    vocab_file=f"{tokenizer_dir}/vocab.json",
    merges_file=f"{tokenizer_dir}/merges.txt",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

hf_tokenizer.save_pretrained("sv_tokenizer_hf")

print("len(tokenizer):", len(hf_tokenizer))
print("Vocab size:", hf_tokenizer.vocab_size)
print("Special tokens:")
print("bos:", hf_tokenizer.bos_token, hf_tokenizer.bos_token_id)
print("eos:", hf_tokenizer.eos_token, hf_tokenizer.eos_token_id)
print("pad:", hf_tokenizer.pad_token, hf_tokenizer.pad_token_id)
print("unk:", hf_tokenizer.unk_token, hf_tokenizer.unk_token_id)

len(tokenizer): 8000
Vocab size: 8000
Special tokens:
bos: <s> 0
eos: </s> 2
pad: <pad> 1
unk: <unk> 3


In [20]:
!ls -lh sv_tokenizer

total 352
-rw-r--r--@ 1 ramneekkaur  staff    62K 23 Apr 11:15 merges.txt
-rw-r--r--@ 1 ramneekkaur  staff   109K 23 Apr 11:15 vocab.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
import os

print(os.path.exists("sv_tokenizer/vocab.json"))
print(os.path.exists("sv_tokenizer/merges.txt"))

True
True


In [22]:
!wc -l sv_tokenizer/merges.txt
!head sv_tokenizer/vocab.json

    7741 sv_tokenizer/merges.txt


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{"<s>":0,"<pad>":1,"</s>":2,"<unk>":3,"!":4,"\"":5,"#":6,"$":7,"%":8,"&":9,"'":10,"(":11,")":12,"*":13,"+":14,",":15,"-":16,".":17,"/":18,"0":19,"1":20,"2":21,"3":22,"4":23,"5":24,"6":25,"7":26,"8":27,"9":28,":":29,";":30,"<":31,"=":32,">":33,"?":34,"@":35,"A":36,"B":37,"C":38,"D":39,"E":40,"F":41,"G":42,"H":43,"I":44,"J":45,"K":46,"L":47,"M":48,"N":49,"O":50,"P":51,"Q":52,"R":53,"S":54,"T":55,"U":56,"V":57,"W":58,"X":59,"Y":60,"Z":61,"[":62,"\\":63,"]":64,"^":65,"_":66,"`":67,"a":68,"b":69,"c":70,"d":71,"e":72,"f":73,"g":74,"h":75,"i":76,"j":77,"k":78,"l":79,"m":80,"n":81,"o":82,"p":83,"q":84,"r":85,"s":86,"t":87,"u":88,"v":89,"w":90,"x":91,"y":92,"z":93,"{":94,"|":95,"}":96,"~":97,"¡":98,"¢":99,"£":100,"¤":101,"¥":102,"¦":103,"§":104,"¨":105,"©":106,"ª":107,"«":108,"¬":109,"®":110,"¯":111,"°":112,"±":113,"²":114,"³":115,"´":116,"µ":117,"¶":118,"·":119,"¸":120,"¹":121,"º":122,"»":123,"¼":124,"½":125,"¾":126,"¿":127,"À":128,"Á":129,"Â":130,"Ã":131,"Ä":132,"Å":133,"Æ":134,"Ç":135,"È":13

In [1]:
with open("train.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(next(f)[:300])
        print("-" * 80)

Den röda fanan kan åsyfta: Den röda fanan (socialism) – en flagga som en traditionell symbol för socialism Bandiera Rossa – en av arbetarrörelsens mest berömda sånger The Red Flag – en kampsång som är stor i den engelska arbetarrörelsen Se även Röda fanans orden – en orden i Sovjetryssland, därefter
--------------------------------------------------------------------------------
Wiedemannia glacicola är en tvåvingeart som beskrevs av Wagner 1985. Wiedemannia glacicola ingår i släktet Wiedemannia och familjen dansflugor. Artens utbredningsområde är Frankrike. Inga underarter finns listade i Catalogue of Life. Källor Dansflugor glacicola

--------------------------------------------------------------------------------
Silva Hakobjan (armeniska: Սիլվա Հակոբյան), född 23 oktober 1988 i Vajk, är en armenisk sångerska. År 2006 vann hon BBC-tävlingen "Next Big Thing". Hakobjan föddes i staden Vajk som ligger i marzen Vajots Dzor i södra Armenien. Hon har framträtt sedan 4 års ålder. År 2006 s

In [2]:
from tokenizers import ByteLevelBPETokenizer

raw_tokenizer = ByteLevelBPETokenizer(
    "sv_tokenizer/vocab.json",
    "sv_tokenizer/merges.txt"
)

print(raw_tokenizer.encode("Sverige är ett land i Europa.").tokens)

['S', 'verig', 'e', 'ĠÃ¤r', 'Ġett', 'Ġland', 'Ġi', 'ĠEuropa', '.']


In [4]:
with open("train.txt", "r", encoding="utf-8") as f:
    for i in range(20):
        line = next(f)
        if "är" in line or "Ã¤r" in line:
            print(line)

Den röda fanan kan åsyfta: Den röda fanan (socialism) – en flagga som en traditionell symbol för socialism Bandiera Rossa – en av arbetarrörelsens mest berömda sånger The Red Flag – en kampsång som är stor i den engelska arbetarrörelsen Se även Röda fanans orden – en orden i Sovjetryssland, därefter Sovjetunionen

Wiedemannia glacicola är en tvåvingeart som beskrevs av Wagner 1985. Wiedemannia glacicola ingår i släktet Wiedemannia och familjen dansflugor. Artens utbredningsområde är Frankrike. Inga underarter finns listade i Catalogue of Life. Källor Dansflugor glacicola

Silva Hakobjan (armeniska: Սիլվա Հակոբյան), född 23 oktober 1988 i Vajk, är en armenisk sångerska. År 2006 vann hon BBC-tävlingen "Next Big Thing". Hakobjan föddes i staden Vajk som ligger i marzen Vajots Dzor i södra Armenien. Hon har framträtt sedan 4 års ålder. År 2006 ställde hon med låten "I Like" upp i BBC-tävlingen "Next Big Thing". Låten komponerades av hennes syster Mane och producerades av hennes bror Edgar.

In [5]:
sample = "Sverige är ett land i Europa."
enc = raw_tokenizer.encode(sample)

print("TOKENS:", enc.tokens)
print("IDS:", enc.ids)
print("DECODED:", raw_tokenizer.decode(enc.ids))

TOKENS: ['S', 'verig', 'e', 'ĠÃ¤r', 'Ġett', 'Ġland', 'Ġi', 'ĠEuropa', '.']
IDS: [54, 720, 72, 299, 478, 900, 274, 2883, 17]
DECODED: Sverige är ett land i Europa.


In [25]:
# from tokenizers import ByteLevelBPETokenizer
# import os

# tokenizer_dir = "sv_tokenizer_fixed"
# os.makedirs(tokenizer_dir, exist_ok=True)

# bpe_tokenizer = ByteLevelBPETokenizer()
# bpe_tokenizer.train(
#     files=["train.txt"],
#     vocab_size=8000,
#     min_frequency=2,
#     special_tokens=["<s>", "</s>", "<unk>", "<pad>"]
# )

# bpe_tokenizer.save_model(tokenizer_dir)
# print(os.listdir(tokenizer_dir))




['merges.txt', 'vocab.json']


In [26]:
# from tokenizers import ByteLevelBPETokenizer

# raw_tokenizer = ByteLevelBPETokenizer(
#     "sv_tokenizer_fixed/vocab.json",
#     "sv_tokenizer_fixed/merges.txt"
# )

# print(raw_tokenizer.encode("Sverige är ett land i Europa.").tokens)

['S', 'verig', 'e', 'ĠÃ¤r', 'Ġett', 'Ġland', 'Ġi', 'ĠEuropa', '.']


In [27]:
# tokenizer_json_path = "sv_tokenizer_fixed/tokenizer.json"
# raw_tokenizer.save(tokenizer_json_path)
# print("Saved:", tokenizer_json_path)

Saved: sv_tokenizer_fixed/tokenizer.json


In [29]:
# from transformers import PreTrainedTokenizerFast

# hf_tokenizer = PreTrainedTokenizerFast(
#     tokenizer_file="sv_tokenizer_fixed/tokenizer.json",
#     bos_token="<s>",
#     eos_token="</s>",
#     unk_token="<unk>",
#     pad_token="<pad>",
# )

# print("len(tokenizer):", len(hf_tokenizer))
# print("bos:", hf_tokenizer.bos_token, hf_tokenizer.bos_token_id)
# print("eos:", hf_tokenizer.eos_token, hf_tokenizer.eos_token_id)
# print("unk:", hf_tokenizer.unk_token, hf_tokenizer.unk_token_id)
# print("pad:", hf_tokenizer.pad_token, hf_tokenizer.pad_token_id)

len(tokenizer): 8000
bos: <s> 0
eos: </s> 1
unk: <unk> 2
pad: <pad> 3


In [30]:
# sample = "Sverige är ett land i norra Europa."
# enc = hf_tokenizer(sample)

# print(enc)
# print(enc["input_ids"])
# print(hf_tokenizer.decode(enc["input_ids"]))

{'input_ids': [54, 720, 72, 299, 478, 900, 274, 1844, 2883, 17], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
[54, 720, 72, 299, 478, 900, 274, 1844, 2883, 17]
Sverige är ett land i norra Europa.


In [8]:
import os
project_dir = "/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task"
checkpoint_dir = os.path.join(project_dir, "checkpoints")
final_model_dir = os.path.join(project_dir, "final_model")
logs_dir = os.path.join(project_dir, "logs")

for d in [checkpoint_dir, final_model_dir, logs_dir]:
    os.makedirs(d, exist_ok=True)

print("Created:")
print(checkpoint_dir)
print(final_model_dir)
print(logs_dir)

Created:
/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/checkpoints
/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/final_model
/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/logs


In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "text",
    data_files={
        "train": "train.txt",
        "validation": "val.txt",
        "test": "test.txt",
    }
)

dataset

/opt/anaconda3/envs/pyg310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2349848
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 23978
    })
    test: Dataset({
        features: ['text'],
        num_rows: 23979
    })
})

In [10]:
def tokenize_function(examples):
    return hf_tokenizer(examples["text"])

In [14]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset

Map: 100%|███████████████████████| 23979/23979 [00:02<00:00, 9713.58 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2349848
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 23978
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 23979
    })
})

In [18]:
tokenized_dataset.save_to_disk("/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/tokenized_dataset")

Saving the dataset (5/5 shards): 100%|█| 2349848/2349848 [00:07<00:00, 329938.72
Saving the dataset (1/1 shards): 100%|█| 23978/23978 [00:00<00:00, 941464.66 exa
Saving the dataset (1/1 shards): 100%|█| 23979/23979 [00:00<00:00, 760056.34 exa


In [15]:
import torch

print("PyTorch version:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

PyTorch version: 2.2.0
MPS built: True
MPS available: True


In [16]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [17]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3).to(device)
z = x @ y

print("Device:", device)
print(z)
print(z.device)

Device: mps
tensor([[ 0.0424,  0.5045, -0.2043],
        [ 2.3119, -2.2446,  0.7717],
        [-0.6832, -0.0369, -0.4082]], device='mps:0')
mps:0


In [19]:
block_size = 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])

    # drop remainder
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True, batch_size=1000)
lm_dataset.save_to_disk("/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/lm_dataset")
lm_dataset

Map: 100%|████████████████████████| 23979/23979 [00:23<00:00, 999.59 examples/s]
Saving the dataset (13/13 shards): 100%|█| 3727924/3727924 [00:17<00:00, 215340.
Saving the dataset (1/1 shards): 100%|█| 38724/38724 [00:00<00:00, 153000.02 exa
Saving the dataset (1/1 shards): 100%|█| 37489/37489 [00:00<00:00, 156264.70 exa


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3727924
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 38724
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 37489
    })
})

In [3]:
from transformers import GPT2Config, GPT2LMHeadModel
import torch
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast.from_pretrained("final_model")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

config = GPT2Config(
    vocab_size=len(hf_tokenizer),
    n_positions=128,
    n_ctx=128,
    n_embd=128,
    n_layer=2,
    n_head=2,
    bos_token_id=hf_tokenizer.bos_token_id,
    eos_token_id=hf_tokenizer.eos_token_id,
    pad_token_id=hf_tokenizer.pad_token_id
)

model = GPT2LMHeadModel(config)
model = model.to(device)

print("Model device:", next(model.parameters()).device)
print("Number of parameters:", model.num_parameters())

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Model device: mps:0
Number of parameters: 1437184


In [4]:
print(torch.backends.mps.is_available())              # should be True
print(next(model.parameters()).device)                # should be mps:0

True
mps:0


In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=False
)

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="checkpoint_dir",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    logging_steps=500,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=100,
    save_total_limit=2,
    report_to="none"
)

In [8]:
from datasets import load_from_disk

lm_dataset = load_from_disk("lm_dataset")
small_train = lm_dataset["train"].select(range(50000))
small_val = lm_dataset["validation"].select(range(5000))

In [9]:
from transformers import Trainer
model = model.to(device)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator
)

In [10]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


ImportError: cannot import name 'DTensor' from 'torch.distributed.tensor' (/opt/anaconda3/envs/pyg310/lib/python3.10/site-packages/torch/distributed/tensor/__init__.py)

In [13]:
import os
import torch
import os
project_dir = "/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task"
final_model_dir = os.path.join(project_dir, "final_model")
os.makedirs(final_model_dir, exist_ok=True)

# Save weights
torch.save(model.state_dict(), os.path.join(final_model_dir, "pytorch_model.bin"))

# Save config
model.config.save_pretrained(final_model_dir)

# Save tokenizer
hf_tokenizer.save_pretrained(final_model_dir)

print("✅ Model saved successfully (manual method)")

✅ Model saved successfully (manual method)


In [14]:
import math

eval_results = trainer.evaluate(lm_dataset["test"])
print(eval_results)

perplexity = math.exp(eval_results["eval_loss"])
print("Test perplexity:", perplexity)

{'eval_loss': 4.863563537597656}
Test perplexity: 129.48480490285996


In [17]:
import os
import json
project_dir = "/Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task"
logs_dir = os.path.join(project_dir, "logs")
eval_file = os.path.join(logs_dir, "test_results.json")

with open(eval_file, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2)

print("Saved evaluation results to:", eval_file)

Saved evaluation results to: /Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/logs/test_results.json


In [18]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=hf_tokenizer
)

prompt = "Sverige är ett land"
outputs = generator(
    prompt,
    max_length=60,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,
    num_return_sequences=3
)

for i, out in enumerate(outputs, 1):
    print(f"\n=== Generated sample {i} ===")
    print(out["generated_text"])

Device set to use mps:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/opt/anaconda3/envs/pyg310/lib/python3.10/site-packages/transformers/pytorch_utils.py:335: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
This is a friendly re


=== Generated sample 1 ===
Sverige är ett landslundning av ett bergskön och som har ett stäppar som sattes ett stort antal än två skön på cirka 31 000 får av den en stor och det är en bäls, är från en könvas med en terrängen. Den finns också på en ö i Finland. Den ligger i distriktet Cyre, i den ekonomiska regionen Tob. Den ligger i den västra delen av landet, km nordväst om huvudstaden Jakarta. Tropiskt monsunklimat råder i trakten. Årsmedeltemperaturen i trakten är °C. Den varmaste månaden är maj, då medeltemperaturen är °C, och den kallaste är °C, och den kallaste är °C, och den kallaste är januari, med °C. Den varmaste månaden är °C, och den kallaste är januari, och den kallaste är april, med °C. Den varmaste månaden är januari, med °C. Genomsnittlig årsnederbörd är juli, med °C. Den varmaste månaden är februari, och den kallaste är januari, med °C. Den varmaste månaden är december, med °C. Den varmaste månaden är januari, och den kallaste är december, med °C. Den varmaste månaden

In [19]:
gen_file = os.path.join(logs_dir, "generated_samples.txt")

with open(gen_file, "w", encoding="utf-8") as f:
    f.write(f"Prompt: {prompt}\n\n")
    for i, out in enumerate(outputs, 1):
        f.write(f"=== Sample {i} ===\n")
        f.write(out["generated_text"] + "\n\n")

print("Saved generated outputs to:", gen_file)

Saved generated outputs to: /Users/ramneekkaur/Documents/Paderborn_University/Project_Ice_breaker_task/logs/generated_samples.txt
